# Map plots

Plot AIS trajectories on the region land map, adapted from `src/plots/plot_mdn.py`.

Positions live in `METER_CRS` (EPSG:25832) — `cords_to_meters` converts them during dataset
construction — so the `land.geojson` background is reprojected to the same CRS before plotting.

Two views:
1. **Coverage** — every trajectory in the dataset, to see spatial extent / density.
2. **Single sample** — one ego track (observed + ground-truth future) with its neighbours.

In [ ]:
import sys
sys.path.insert(0, "src")  # .env sets PYTHONPATH=src, but the kernel may not load it

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
import math

from collections import Counter
from tqdm import tqdm
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader

from data.graph.build_dataloader import graph_loader
from utils.config import DATA_FOLDER_PATH, METER_CRS

SOURCE = "fh"
REGION = "kiel"


def load_land(region=REGION, source=SOURCE):
    """Region land polygons reprojected to METER_CRS (same frame as the trajectories)."""
    path = DATA_FOLDER_PATH / f"maps/2_standardized/{source}_10/{region}/land.geojson"
    return gpd.read_file(path).to_crs(METER_CRS)


land = load_land()
land.crs


In [ ]:
# Load the whole dataset for the region (this reads the parquet files and builds the
# graph edges, so the first run takes a while; graph_loader caches it afterwards).
loader, dset = graph_loader(
    data_folder=DATA_FOLDER_PATH / f"ais/4_features/{SOURCE}_10/{REGION}",
    flag="test",
    min_date=pd.Timestamp("2022-01-01"),
    max_date=pd.Timestamp("2024-01-01"),
    batch_size=1,
    pred_len=30,
    obs_len=60,
    max_edge_dist=500,
    shuffle=False,
    ship_group="all",
)
print(f"samples: {len(dset)} | trajectories: {len(dset.pos_map)}")


In [ ]:
14407680 / (14407680 + 2011715 + 3674634)

In [ ]:
(14407680 + 2011715 + 3674634)

In [ ]:
2011715 / (14407680 + 2011715 + 3674634)

In [ ]:
3674634 / (14407680 + 2011715 + 3674634)

## 1. Coverage map

Every trajectory in `dset.pos_map`, drawn as a thin translucent line so overlap reads as density.
`pos_map` arrays are zero-padded by `[obs_len … pred_len]`, so we slice off the padding first.

In [ ]:
def plot_coverage(dset, land, region=REGION, color="navy", lw=0.4, alpha=0.15, figsize=(10, 10)):
    """All dataset trajectories over the land map (LineCollection for speed)."""
    obs_len, pred_len = dset.obs_len, dset.pred_len
    segments = []
    for pos in dset.pos_map.values():
        real = pos[obs_len:-pred_len, :2]  # strip the [obs_len … pred_len] zero padding
        if len(real) >= 2:
            segments.append(real)

    fig, ax = plt.subplots(figsize=figsize)
    land.plot(ax=ax, facecolor="lightgray", edgecolor="black", alpha=0.5)
    ax.add_collection(LineCollection(segments, colors=color, linewidths=lw, alpha=alpha))
    ax.autoscale()
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"AIS trajectory coverage — {region} ({len(segments)} tracks)")
    return fig, ax


fig, ax = plot_coverage(dset, land)
plt.show()


In [ ]:
"""Snippet: overlay NEREUS MDN predictions on the region map.

Copy these blocks into map_plots.ipynb (they assume the earlier cells already ran, i.e.
`dset`, `land`, `REGION`, `SOURCE`, `DATA_FOLDER_PATH`, `np`, `plt` exist). The model's
_build_model hardcodes CUDA, so a GPU is required (same as full_eval_nereus.py).
"""

# ── cell: imports + config ────────────────────────────────────────────────────
from pathlib import Path

import matplotlib.colors as mcolors
import torch
from matplotlib import cm
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from torch_geometric.data import Batch

from data.map.rasterize import Rasterizer
from data.map.scene_gernerator import SceneLoader
from train.pl_modules import NereusModule
from utils.config import STEPS_PER_MINUTE

DE_NORMALIZE = 100  # normalized rel. displacement -> meters (matches full_eval_nereus.py)

# Region bounding boxes (lon_min, lat_min, lon_max, lat_max), from full_eval_nereus.py.
ALL_REGIONS = {
    "kiel":        [10.12, 54.31, 10.33, 54.46],
    "aarhus":      [10.21, 56.04, 10.47, 56.17],
    "odense":      [10.42, 55.42, 10.68, 55.55],
    "little_belt": [9.64,  55.25,  9.90, 55.37],
}


# ── cell: model loading ───────────────────────────────────────────────────────
def swap_rasterizer(model, bbox):
    """Point the model (and its map/prior CNNs) at this region's rasterizer."""
    rasterizer = Rasterizer(bbox)
    model.rasterizer = rasterizer
    if hasattr(model, "map_cnn") and model.map_cnn is not None:
        model.map_cnn.rasterizer = rasterizer
    if hasattr(model, "prior_cnn") and model.prior_cnn is not None:
        model.prior_cnn.rasterizer = rasterizer
    return rasterizer


def load_model(ckpt_dir, bbox, region, device, source="fh"):
    """Load a Lightning NereusModule checkpoint and build the region scene tensor."""
    ckpt_path = sorted(Path(ckpt_dir).rglob("*.ckpt"))[0]  # e.g. version_15/best.ckpt
    pl_module = NereusModule.load_from_checkpoint(str(ckpt_path), map_location=device)
    model = pl_module.model.to(device)
    model.eval()
    swap_rasterizer(model, bbox)

    map_folder = DATA_FOLDER_PATH / f"maps/2_standardized/{source}_10/{region}"
    scene = torch.from_numpy(
        np.ascontiguousarray(SceneLoader(Rasterizer(bbox)).load_scene(map_folder))
    ).to(device, torch.float32)
    return pl_module, model, scene


# ── cell: inference on one dataset sample ─────────────────────────────────────
def predict(pl_module, model, scene, data, device):
    """Run the MDN on one dataset item; return the ego's numpy trajectories.

    Returns dict with:
      pred_abs   [T, 2]     expected (pi-weighted) absolute trajectory
      pred_abs_k [K, T, 2]  per-mode absolute trajectories
      pi_k       [K]        time-averaged mode probabilities (for coloring)
    """
    cfg = pl_module.cfg
    T, K = cfg.pred_len, cfg.mdn_modes
    batch = Batch.from_data_list([data]).to(device)  # adds the `batch` vector the GNN needs

    with torch.inference_mode():
        ego_idx = batch.is_ego.nonzero(as_tuple=True)[0]
        B = ego_idx.numel()
        mdn_out = model(batch, scene).view(B, T, K, 5)

        pi = torch.softmax(mdn_out[..., 0], dim=-1)          # [B, T, K]
        mu = mdn_out[..., 1:3]                                # [B, T, K, 2]
        last = batch.x_pos[ego_idx, -1:, :]                  # [B, 1, 2]

        exp_rel = (pi.unsqueeze(-1) * mu).sum(dim=2)         # [B, T, 2]
        pred_abs = torch.cumsum(exp_rel, dim=1) * DE_NORMALIZE + last
        mu_k = mu.permute(0, 2, 1, 3)                        # [B, K, T, 2]
        pred_abs_k = torch.cumsum(mu_k, dim=2) * DE_NORMALIZE + last.unsqueeze(1)
        pi_k = pi.mean(dim=1)                                # [B, K]

    return {
        "pred_abs":   pred_abs[0].cpu().numpy(),
        "pred_abs_k": pred_abs_k[0].cpu().numpy(),
        "pi_k":       pi_k[0].cpu().numpy(),
    }


# ── cell: plotting (extends plot_sample with predictions) ─────────────────────
# Per-series styles used when color_by_time=True: color encodes time, so series are
# told apart by (linestyle, linewidth) only — the K modes deliberately share one style.
_TIME_STYLES = {
    "observed":     dict(ls="-",  lw=1.0, marker="o", ms=3),
    "ground_truth": dict(ls="-",  lw=3.0),
    "expected":     dict(ls="--", lw=2.0),
    "modes":        dict(ls=":",  lw=1.2),
}


def _add_time_line(ax, xy, cmap, norm, zorder=2, **style):
    """Draw xy [L,2] as a line whose color encodes prediction time (minutes)."""
    xy = np.asarray(xy)
    if len(xy) < 2:
        return
    minutes = np.arange(1, len(xy) + 1) / STEPS_PER_MINUTE
    segments = np.concatenate([xy[:-1, None], xy[1:, None]], axis=1)
    lc = LineCollection(segments, cmap=cmap, norm=norm, zorder=zorder,
                        linestyles=style.get("ls", "-"), linewidths=style.get("lw", 1.5))
    lc.set_array(minutes[1:])  # color each segment by its end time
    ax.add_collection(lc)


def plot_sample_with_pred(data, land, pred, region=None, pad=300, figsize=(8, 8),
                          color_by_time=False, show_colorbar=True):
    """plot_sample + predictions.

    color_by_time=False: modes colored by their probability pi (as in plot_mdn.py).
    color_by_time=True:  every trajectory colored by prediction timestep (shared colorbar),
                         and the series are distinguished by linestyle/thickness/marker
                         instead of color (the 3 modes intentionally look identical).
    show_colorbar:       draw the probability / time colorbar (turn off for compact panels).
    Axes are forced square (equal data span + equal aspect) so every plot has the same shape.
    """
    region = region or REGION
    ego_obs = data.x_pos[0].numpy()[data.x_mask[0].numpy()]
    ego_fut = data.y_pos[0].numpy()[data.y_mask[0].numpy()]
    pred_abs, pred_abs_k, pi_k = pred["pred_abs"], pred["pred_abs_k"], pred["pi_k"]

    fig, ax = plt.subplots(figsize=figsize)
    land.plot(ax=ax, facecolor="lightgray", edgecolor="black", alpha=0.5)

    # neighbours (observed) — always neutral gray
    for i in range(1, data.x_pos.shape[0]):
        nb = data.x_pos[i].numpy()[data.x_mask[i].numpy()]
        if len(nb):
            ax.plot(nb[:, 0], nb[:, 1], color="gray", lw=1.0, alpha=0.5,
                    label=None if color_by_time else ("Neighbours" if i == 1 else None))

    if color_by_time:
        T = pred_abs.shape[0]
        cmap = cm.viridis
        norm = mcolors.Normalize(vmin=0.0, vmax=T / STEPS_PER_MINUTE)

        # observed history: neutral reference (it's the past, not the predicted horizon)
        ax.plot(ego_obs[:, 0], ego_obs[:, 1], color="black", alpha=0.7, zorder=1,
                **_TIME_STYLES["observed"])
        # future series: color = time, differentiated only by line style / thickness
        _add_time_line(ax, ego_fut, cmap, norm, zorder=4, **_TIME_STYLES["ground_truth"])
        _add_time_line(ax, pred_abs, cmap, norm, zorder=3, **_TIME_STYLES["expected"])
        for k in range(pred_abs_k.shape[0]):
            _add_time_line(ax, pred_abs_k[k], cmap, norm, zorder=2, **_TIME_STYLES["modes"])

        if show_colorbar:
            plt.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                         label="Prediction horizon [min]")
        handles = [
            Line2D([0], [0], color="black", label="Observed", **_TIME_STYLES["observed"]),
            Line2D([0], [0], color="black", label="Ground truth", **_TIME_STYLES["ground_truth"]),
            Line2D([0], [0], color="black", label="Expected", **_TIME_STYLES["expected"]),
            Line2D([0], [0], color="black", label=f"{pred_abs_k.shape[0]} modes", **_TIME_STYLES["modes"]),
            Line2D([0], [0], color="gray", lw=1.0, label="Neighbours"),
        ]
        ax.legend(handles=handles, loc="best", fontsize=8)
    else:
        # ego observed + ground truth
        ax.scatter(ego_obs[:, 0], ego_obs[:, 1], color="blue", s=8, alpha=0.8, label="Observed")
        ax.scatter(ego_fut[:, 0], ego_fut[:, 1], color="green", s=8, alpha=0.8, label="Ground truth")

        # K predicted modes, colored by mode probability (as in plot_mdn.py)
        cmap, norm = cm.plasma, mcolors.Normalize(vmin=0.0, vmax=1.0)
        for k in range(pred_abs_k.shape[0]):
            traj = pred_abs_k[k]
            ax.plot(traj[:, 0], traj[:, 1], color=cmap(norm(pi_k[k])), lw=1.5, alpha=0.9,
                    zorder=2, label=f"Mode {k} (π={pi_k[k]:.2f})")
        ax.plot(pred_abs[:, 0], pred_abs[:, 1], color="red", lw=2.0, ls="--", zorder=3,
                label="Expected")
        if show_colorbar:
            plt.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, label="Mode probability (π)")
        ax.legend(loc="best", fontsize=8)

    # Square window: center on the data, use the larger of the two spans for both axes.
    all_xy = np.concatenate([ego_obs, ego_fut, pred_abs, pred_abs_k.reshape(-1, 2)])
    (minx, miny), (maxx, maxy) = all_xy.min(0), all_xy.max(0)
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max(maxx - minx, maxy - miny) / 2 + pad
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.set_title(f"MDN prediction — {region} (id={data.target_id})")
    ax.grid(True, linestyle=":", linewidth=0.7, alpha=0.6)
    return fig, ax



def gt_geometry_batch(y_pos, y_mask):
    """Vectorized gt_geometry over a batch.

    y_pos [G, T, 2], y_mask [G, T] bool (a contiguous prefix per row).
    Returns a dict of [G] tensors. Padding steps are (0, 0), so everything is masked.
    """
    p = y_pos.double()
    m = y_mask.bool()
    G = p.shape[0]
    idx = torch.arange(G, device=p.device)
    counts = m.sum(1)                                    # valid length per row

    seg = p[:, 1:] - p[:, :-1]                           # [G, T-1, 2]
    seg_m = m[:, 1:] & m[:, :-1]                          # segment valid iff both ends valid
    path_len = (seg.norm(dim=-1) * seg_m).sum(1)         # [G]

    first = p[:, 0]                                      # prefix mask -> index 0 is valid when counts>0
    last = p[idx, (counts - 1).clamp(min=0)]             # last valid position
    chord = (last - first).norm(dim=-1)

    # total absolute turning angle (wrapped per step)
    ang = torch.atan2(seg[..., 1], seg[..., 0])          # [G, T-1]
    dang = (ang[:, 1:] - ang[:, :-1] + math.pi) % (2 * math.pi) - math.pi
    dang_m = seg_m[:, 1:] & seg_m[:, :-1]
    turn = (dang.abs() * dang_m).sum(1)                  # radians

    # max perpendicular deviation from the start->end chord
    d = (last - first) / chord.clamp(min=1e-6).unsqueeze(-1)     # [G, 2] unit chord dir
    rel = p - first.unsqueeze(1)                                 # [G, T, 2]
    perp = rel - (rel * d.unsqueeze(1)).sum(-1, keepdim=True) * d.unsqueeze(1)
    max_dev = perp.norm(dim=-1).masked_fill(~m, 0.0).max(dim=1).values

    sinuosity = torch.where(chord > 1e-6, path_len / chord.clamp(min=1e-6),
                            torch.ones_like(chord))
    turn = turn.masked_fill(counts < 3, 0.0)             # need >=2 segments to turn
    max_dev = max_dev.masked_fill(counts < 2, 0.0)

    return {
        "n": counts, "path_len": path_len, "chord": chord,
        "sinuosity": sinuosity, "turn_deg": torch.rad2deg(turn), "max_dev": max_dev,
    }

def geometry_dataframe(dset, batch_size=512, num_workers=4, device=None):
    """Iterate the dataset once (shuffle=False, drop_last=False) and return a DataFrame of
    per-sample GT geometry. Accepts a full dataset or a torch Subset.

    `index` is the position in the passed `dset` (so `dset[index]` works); `base_index`
    is the index into the underlying dataset (for gt_future / the full dataset)."""
    loader = DataLoader(dset, batch_size=batch_size, shuffle=False, drop_last=False,
                        num_workers=num_workers)
    device = device or torch.device("cpu")
    frames, start = [], 0
    for batch in tqdm(loader, desc="gt_geometry"):
        geo = gt_geometry_batch(batch.y_pos.to(device), batch.y_mask.to(device))
        df = pd.DataFrame({k: v.cpu().numpy() for k, v in geo.items()})
        df.insert(0, "index", np.arange(start, start + len(df)))
        frames.append(df)
        start += len(df)
    df = pd.concat(frames, ignore_index=True)

    base, base_map = _resolve_base(dset)  # handles Subset (or plain dataset -> identity map)
    df.insert(1, "base_index", [base_map[i] for i in df["index"]])
    df["target_id"] = [base.items[b][1] for b in df["base_index"]]  # avoids relying on PyG collation
    return df


def _resolve_base(dset):
    """Unwrap (possibly nested) torch Subsets -> (base_dataset, map) where
    map[pos] is the base-dataset index of the pos-th sample of `dset`."""
    idx = list(range(len(dset)))
    base = dset
    while isinstance(base, Subset):
        idx = [base.indices[j] for j in idx]
        base = base.dataset
    return base, idx

def _fit_metrics(pred_abs, pred_abs_k, y_pos, y_mask):
    """Per-sample displacement metrics for a batch. Shapes: pred_abs [G,T,2],
    pred_abs_k [G,K,T,2], y_pos [G,T,2], y_mask [G,T]. Returns dict of [G] tensors."""
    G, K = pred_abs_k.shape[0], pred_abs_k.shape[1]
    m = y_mask.float()
    counts = y_mask.sum(1)
    last = (counts - 1).clamp(min=0)
    gi = torch.arange(G, device=pred_abs.device)

    de = (pred_abs - y_pos).norm(dim=-1)                          # [G,T]
    ade = (de * m).sum(1) / m.sum(1).clamp(min=1)
    fde = de[gi, last]                                            # error at last valid step
    de_k = (pred_abs_k - y_pos.unsqueeze(1)).norm(dim=-1)         # [G,K,T]
    ade_k = (de_k * m.unsqueeze(1)).sum(2) / m.sum(1, keepdim=True).clamp(min=1)   # [G,K]
    fde_k = de_k.gather(2, last.view(G, 1, 1).expand(G, K, 1)).squeeze(2)          # [G,K]
    return dict(ade=ade, fde=fde, min_ade_k=ade_k.min(1).values,
                min_fde_k=fde_k.min(1).values, n=counts)


def predict_dataframe(subset, pl_module, model, scene, device,
                      batch_size=256, num_workers=4, store_pred=True):
    """Run `model` over a subset (batched) and return predictions + fit metrics per sample.

    Same keys as geometry_dataframe (`index`/`base_index`/`target_id`) so the two merge.
    Metric columns: ade, fde, min_ade_k, min_fde_k (meters), n (valid horizon length).
    With store_pred, adds object columns pred_abs [T,2], pred_abs_k [K,T,2], pi_k [K]."""
    cfg = pl_module.cfg
    T, K = cfg.pred_len, cfg.mdn_modes
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, drop_last=False,
                        num_workers=num_workers)
    base, base_map = _resolve_base(subset)

    model.eval()
    frames, start = [], 0
    with torch.inference_mode():
        for batch in tqdm(loader, desc="predict"):
            batch = batch.to(device)
            ego_idx = batch.is_ego.nonzero(as_tuple=True)[0]
            G = ego_idx.numel()

            mdn_out = model(batch, scene).view(G, T, K, 5)
            pi = torch.softmax(mdn_out[..., 0], dim=-1)          # [G,T,K]
            mu = mdn_out[..., 1:3]                                # [G,T,K,2]
            last = batch.x_pos[ego_idx, -1:, :]                  # [G,1,2]
            exp_rel = (pi.unsqueeze(-1) * mu).sum(2)             # [G,T,2]
            pred_abs = torch.cumsum(exp_rel, 1) * DE_NORMALIZE + last
            mu_k = mu.permute(0, 2, 1, 3)                        # [G,K,T,2]
            pred_abs_k = torch.cumsum(mu_k, 2) * DE_NORMALIZE + last.unsqueeze(1)
            pi_k = pi.mean(1)                                    # [G,K]

            metrics = _fit_metrics(pred_abs, pred_abs_k, batch.y_pos, batch.y_mask)
            df = pd.DataFrame({k: v.cpu().numpy() for k, v in metrics.items()})
            df.insert(0, "index", np.arange(start, start + G))
            if store_pred:
                df["pred_abs"] = list(pred_abs.cpu().numpy().astype("float32"))
                df["pred_abs_k"] = list(pred_abs_k.cpu().numpy().astype("float32"))
                df["pi_k"] = list(pi_k.cpu().numpy().astype("float32"))
            frames.append(df)
            start += G

    out = pd.concat(frames, ignore_index=True)
    out.insert(1, "base_index", [base_map[i] for i in out["index"]])
    out["target_id"] = [base.items[b][1] for b in out["base_index"]]
    return out

def pareto_front(df, x="ade", y="turn_deg", minimize_x=True, maximize_y=True):
    """Non-dominated rows for two objectives. Default: minimize x (ade), maximize y (turn_deg).

    A row is Pareto-optimal if no other row is at least as good in both objectives and
    strictly better in at least one. O(n log n) skyline sweep. Non-finite rows are dropped.
    Returns the optimal rows sorted along x.
    """
    sub = df[np.isfinite(df[x]) & np.isfinite(df[y])]
    sx = sub[x].to_numpy(float) * (1 if minimize_x else -1)
    sy = sub[y].to_numpy(float) * (1 if maximize_y else -1)
    order = np.lexsort((-sy, sx))  # x ascending, ties broken by y descending
    keep, best_y = [], -np.inf
    for i in order:
        if sy[i] > best_y:          # nothing with smaller/equal x has a higher y -> optimal
            keep.append(i)
            best_y = sy[i]
    return sub.iloc[keep].sort_values(x, ascending=minimize_x)


def plot_pareto(df, front, x="ade", y="turn_deg", figsize=(7, 6)):
    """Scatter all samples + the Pareto skyline staircase."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(df[x], df[y], s=8, color="lightgray", label="all samples")
    f = front.sort_values(x)
    ax.step(f[x], f[y], where="post", color="crimson", lw=1.5, zorder=2)
    ax.scatter(f[x], f[y], s=28, color="crimson", zorder=3, label="Pareto-optimal")
    ax.set_xlabel(f"{x}  (lower is better)")
    ax.set_ylabel(f"{y}  (higher is better)")
    ax.set_title(f"Pareto skyline — {len(front)} optimal of {len(df)}")
    ax.legend()
    return fig, ax


def plot_df_rows(rows, dset, land, n=None, index_col="base_index", **plot_kwargs):
    """Plot each row's stored prediction on the map (no model call).

    Rebuilds the `pred` dict from the pred_abs / pred_abs_k / pi_k columns and indexes
    `dset` by `index_col` (use "base_index" with the full dataset, "index" with the
    same object passed to predict_dataframe). Extra kwargs (color_by_time, show_colorbar,
    pad, ...) pass through to plot_sample_with_pred.
    """
    rows = rows if n is None else rows.head(n)
    figs = []
    for _, row in rows.iterrows():
        data = dset[int(row[index_col])]
        pred = {
            "pred_abs":   np.asarray(row["pred_abs"]),
            "pred_abs_k": np.asarray(row["pred_abs_k"]),
            "pi_k":       np.asarray(row["pi_k"]),
        }
        fig, ax = plot_sample_with_pred(data, land, pred, **plot_kwargs)
        title = [f"id={data.target_id}"]
        if "ade" in row:
            title.append(f"ADE={row['ade']:.1f}m")
        if "turn_deg" in row:
            title.append(f"turn={row['turn_deg']:.0f}°")
        ax.set_title("  ".join(title))
        figs.append((fig, ax))
        plt.show()
    return figs

# e.g. filter curved samples, then evaluate the model on just those:
#   curved_idx = geom.index[(geom.path_len >= 150) & (geom.turn_deg >= 40)].tolist()
#   sub = torch.utils.data.Subset(dset, curved_idx)
#   pred_df = predict_dataframe(sub, pl_module, model, scene, device)
#   report = pred_df.merge(geom, on="base_index", suffixes=("", "_geom"))
#pred_df = predict_dataframe(dset, pl_module, model, scene, device)
#pred_df[["index", "base_index", "target_id", "ade", "fde", "min_ade_k", "min_fde_k", "n"]].describe()





In [ ]:
# ── cell: run it ──────────────────────────────────────────────────────────────
device = torch.device("cuda:0")
bbox = ALL_REGIONS[REGION]
pl_module, model, scene = load_model(
    "lightning_logs/nereus_ablation/version_15", bbox, REGION, device, source=SOURCE
)



In [ ]:

counter = Counter([dset[i].x_pos.shape[0] for i in tqdm(range(len(dset)))])
#counter = Counter([dset[i].x_pos.shape[0] for i in tqdm(range(100000))])
counter

# Counter({1: 2358286,
#          2: 841781,
#          3: 283761,
#          4: 102324,
#          5: 39356,
#          6: 17218,
#          7: 8470,
#          8: 4827,
#          9: 3432,
#          10: 2339,
#          11: 2189,
#          12: 1735,
#          13: 1345,
#          14: 1072,
#          15: 945,
#          16: 774,
#          21: 720,
#          18: 610,
#          17: 588,
#          22: 585,
#          20: 523,
#          19: 513,
#          23: 413,
#          24: 332,
#          25: 210,
#          26: 170,
#          27: 95,
#          28: 20,
#          29: 1})



In [ ]:
neighbor_list = [(i,dset[i].x_pos.shape[0]) for i in tqdm(range(len(dset))) if dset[i].x_pos.shape[0] > 1]

many_neighbors = np.random.choice([x[0] for x in filter(lambda x: x[1] > 22, neighbor_list)],100,replace=False)



subset = Subset(dset, [x[0] for x in neighbor_list])

In [ ]:
geom = geometry_dataframe(subset)


In [ ]:
geom

In [ ]:

geom = geometry_dataframe(subset)
geom = geom[geom["n"] == 30]
geom = geom[geom["turn_deg"] > 60]

subset2 = Subset(subset, geom["index"].to_list())
df = predict_dataframe(subset2, pl_module, model, scene, device,batch_size=512)
df_pred = df.join(geom.set_index("base_index"),on="base_index",rsuffix="_geom")
df_pareto = pareto_front(df_pred)

In [ ]:
import pandas as pd
import polars as pl
from collections import Counter
from utils.config import SHIP_DB_PATH


def samples_per_ship_group(dataset, nodes_path):
    """Samples, trajectories, and average speed per ship_group.

    Rebuilds the traj_id -> ship_group map exactly like GraphDataset.__init__:
    (mmsi, traj_id) pairs from the nodes parquet joined with the ship DB.
    Speed is averaged over all position reports of trajectories in the dataset.
    """
    nodes = pl.read_parquet(nodes_path)
    traj_map = (
        nodes.select(["mmsi", "traj_id"]).unique()
        .join(pl.read_parquet(SHIP_DB_PATH).select(["mmsi", "ship_group"]),
              on="mmsi", how="left")
        .with_columns(pl.col("ship_group").fill_null("unknown"))
    )
    group_of = {tid: sg for tid, sg in traj_map.select(["traj_id", "ship_group"]).rows()}

    ds_traj_ids = {tid for _, tid in dataset.items}
    sample_counts = Counter(group_of.get(tid, "unknown") for _, tid in dataset.items)
    traj_counts   = Counter(group_of.get(tid, "unknown") for tid in ds_traj_ids)

    # speed = (
    #     nodes.filter(pl.col("traj_id").is_in(list(ds_traj_ids)))
    #     .join(traj_map.select(["traj_id", "ship_group"]).unique("traj_id"), on="traj_id")
    #     .group_by("ship_group")
    #     .agg(pl.col("speed").mean().alias("mean_speed"))
    # )
    speed = (
            nodes.filter(pl.col("traj_id").is_in(list(ds_traj_ids)))
            .join(traj_map.select(["traj_id", "ship_group"]).unique("traj_id"), on="traj_id")
            .group_by(["ship_group", "traj_id"])
            .agg(pl.col("speed").mean())          # 1) mean per trajectory
            .group_by("ship_group")
            .agg(pl.col("speed").mean().alias("mean_speed"))  # 2) mean over trajectories
        )
    mean_speed = dict(speed.rows())

    df = pd.DataFrame({
        "n_samples":  pd.Series(sample_counts),
        "n_trajs":    pd.Series(traj_counts),
        "mean_speed": pd.Series(mean_speed),
    })
    df[["n_samples", "n_trajs"]] = df[["n_samples", "n_trajs"]].fillna(0).astype(int)
    df["sample_share"] = (df["n_samples"] / df["n_samples"].sum() * 100).round(1)
    df["mean_speed"] = df["mean_speed"].round(2)
    return df.sort_values("n_samples", ascending=False)



In [ ]:


flag = "test"
#AIS_SOURCE = "fh"
#DATA_FOLDER_PATH = Path("data/projects/ship_tracker/assets")
#AIS_FOLDER_PATH = DATA_FOLDER_PATH / "ais/4_features/fh_10/kiel"
#file_name = f"{AIS_SOURCE}_{DATA_FOLDER_PATH.name}_{flag}"
#nodes_path = DATA_FOLDER_PATH / f"{file_name}_ship_features.parquet",
nodes_path = f"data/projects/ship_tracker/assets/ais/4_features/fh/kiel/fh_kiel_{flag}_ship_features.parquet"


counts = samples_per_ship_group(dset, nodes_path)
print(counts)


In [ ]:
df_pred

In [ ]:
df_pareto = pareto_front(df_pred,y="max_dev")

In [ ]:
index_col = "base_index"
plot_kwargs = {}
figs = []
for _, row in df_pareto.iterrows():
    data = dset[int(row[index_col])]
    pred = {
        "pred_abs":   np.asarray(row["pred_abs"]),
        "pred_abs_k": np.asarray(row["pred_abs_k"]),
        "pi_k":       np.asarray(row["pi_k"]),
    }
    fig, ax = plot_sample_with_pred(data, land, pred, **plot_kwargs)
    title = [f"id={data.target_id}"]
    if "ade" in row:
        title.append(f"ADE={row['ade']:.1f}m")
    if "turn_deg" in row:
        title.append(f"turn={row['turn_deg']:.0f}°")
    ax.set_title("  ".join(title))

    print(row["base_index"])
    #plt.savefig(f"map_figures_pareto_dev/test_{row['base_index']}.png")
    plt.show()

    figs.append((fig, ax))



In [ ]:
from utils.config import STEP_SIZE


def _traj_datetime(dset, base_index):
    """Return the observation timestamp of a dataset sample as a pd.Timestamp."""
    cur_t, _ = dset.items[base_index]
    return pd.Timestamp(cur_t * STEP_SIZE, unit="s")


def draw_traj_ax(ax, data, land, pred, title=None, pad=300, show_legend=False):
    """Draw one trajectory (+ MDN prediction) onto an existing Axes.

    Mirrors plot_sample_with_pred but works on a caller-supplied axis so it can
    be embedded in multi-panel figures.
    """
    ego_obs = data.x_pos[0].numpy()[data.x_mask[0].numpy()]
    ego_fut = data.y_pos[0].numpy()[data.y_mask[0].numpy()]
    pred_abs   = pred["pred_abs"]    # [T, 2]
    pred_abs_k = pred["pred_abs_k"]  # [K, T, 2]
    pi_k       = pred["pi_k"]        # [K]

    land.plot(ax=ax, facecolor="lightgray", edgecolor="black", alpha=0.5)

    for i in range(1, data.x_pos.shape[0]):
        nb = data.x_pos[i].numpy()[data.x_mask[i].numpy()]
        if len(nb):
            ax.plot(nb[:, 0], nb[:, 1], color="gray", lw=0.8, alpha=0.5,
                    label="Neighbours" if (show_legend and i == 1) else None)

    ax.scatter(ego_obs[:, 0], ego_obs[:, 1], color="blue", s=5, alpha=0.8,
               label="Observed" if show_legend else None)
    ax.scatter(ego_fut[:, 0], ego_fut[:, 1], color="green", s=5, alpha=0.8,
               label="Ground truth" if show_legend else None)

    cmap_m = cm.plasma
    norm_m = mcolors.Normalize(vmin=0.0, vmax=1.0)
    for k in range(pred_abs_k.shape[0]):
        ax.plot(pred_abs_k[k, :, 0], pred_abs_k[k, :, 1],
                color=cmap_m(norm_m(pi_k[k])), lw=1.0, alpha=0.85, zorder=2,
                label=f"Mode {k} (π={pi_k[k]:.2f})" if show_legend else None)
    ax.plot(pred_abs[:, 0], pred_abs[:, 1], color="red", lw=1.5, ls="--", zorder=3,
            label="Expected" if show_legend else None)

    all_xy = np.concatenate([ego_obs, ego_fut, pred_abs, pred_abs_k.reshape(-1, 2)])
    (minx, miny), (maxx, maxy) = all_xy.min(0), all_xy.max(0)
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    half = max(maxx - minx, maxy - miny) / 2 + pad
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)

    if title:
        ax.set_title(title, fontsize=7.5, pad=3)
    if show_legend:
        ax.legend(fontsize=6, loc="best", markerscale=0.8)





def plot_trajectory_grid(rows, dset, land, nrows=2, ncols=4,
                         index_col="base_index", figsize=None,
                         pad=300, hspace=0.15, suptitle=None):
    """2×4 (or nrows×ncols) trajectory grid.

    Legend is placed in the top-right panel; a shared mode-probability
    colorbar is added to the right of the whole figure.
    """
    rows = rows.head(nrows * ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             #sharex="all", sharey="all",
                             figsize=figsize or (ncols * 4, nrows * 4), gridspec_kw={"hspace": hspace})
    axes = np.array(axes).reshape(nrows, ncols)

    for idx, (_, row) in enumerate(rows.iterrows()):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]

        data = dset[int(row[index_col])]
        pred = {
            "pred_abs":   np.asarray(row["pred_abs"]),
            "pred_abs_k": np.asarray(row["pred_abs_k"]),
            "pi_k":       np.asarray(row["pi_k"]),
        }

        dt = _traj_datetime(dset, int(row[index_col]))

        title = dt.strftime("%Y-%m-%d  %H:%M UTC")
        #line1 = dt.strftime("%Y-%m-%d  %H:%M UTC")
        #parts = []
        #if "ade" in row and pd.notna(row["ade"]):
        #    parts.append(f"ADE={row['ade']:.0f} m")
        #if "turn_deg" in row and pd.notna(row["turn_deg"]):
        #    parts.append(f"turn={row['turn_deg']:.0f}°")
        #line2 = "  ".join(parts)
        #title = f"{line1}\n{line2}" if line2 else line1

        # legend only on top-right panel
        draw_traj_ax(ax, data, land, pred, title=title, pad=pad,
                     show_legend=(r == 0 and c == ncols - 1))

    for idx in range(len(rows), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].set_visible(False)

    # # shared colorbar on the right (steals space from all panels automatically)
    # sm = cm.ScalarMappable(cmap=cm.plasma, norm=mcolors.Normalize(vmin=0.0, vmax=1.0))
    # sm.set_array([])
    # fig.colorbar(sm, ax=axes.ravel().tolist(),
    #              label="Mode probability (π)", shrink=0.6, pad=0.02)

    # if suptitle:
    #     fig.suptitle(suptitle, fontsize=12)

    # return fig, axes

    # shared colorbar — must come before suptitle
    sm = cm.ScalarMappable(cmap=cm.plasma, norm=mcolors.Normalize(vmin=0.0, vmax=1.0))
    sm.set_array([])
    fig.colorbar(sm, ax=axes.ravel().tolist(),
                 label="Mode probability (π)", shrink=0.6, pad=0.01)

    if suptitle:
        visible = [ax for ax in axes.ravel() if ax.get_visible()]
        x0 = min(ax.get_position().x0 for ax in visible)
        x1 = max(ax.get_position().x1 for ax in visible)
        y1 = max(ax.get_position().y1 for ax in visible)
        fig.suptitle(suptitle, fontsize=12, ha="center",
                     x=(x0 + x1) / 2,   # centered over subplots, not full figure
                     y=y1 + 0.05)        # just above the top row

    return fig, axes


In [ ]:
# Example: Pareto-optimal samples in a 2×4 grid.
# df_pareto must already exist (run the geometry + predict cells above first).
df_chosen_traj = df_pareto[df_pareto["base_index"].isin([2577241,2048713,1870201,49147,3386299,3350627,2367963,370770,1854238])]
#

fig, axes = plot_trajectory_grid(
    df_chosen_traj,
    dset=dset,
    land=land,
    nrows=3, ncols=3,
    pad=300,
    suptitle="Example trajectories",
    hspace=-0.3,
    #figsize=(24, 10)
)
plt.show()
fig.savefig("figures/trajectory_grid.pdf", bbox_inches="tight")


In [ ]:
import polars as pl
from utils.config import SHIP_DB_PATH

traj_map = (
    pl.read_parquet(nodes_path)
    .select(["mmsi", "traj_id"]).unique()
    .join(pl.read_parquet(SHIP_DB_PATH).select(["mmsi", "ship_group"]),
          on="mmsi", how="left")
)
group_of = {tid: (sg or "unknown")
            for tid, sg in traj_map.select(["traj_id", "ship_group"]).rows()}


dt_list = []
num_neighbors = []
ship_types = []
for i in many_neighbors:
    dt = _traj_datetime(dset, i)
    dt_list.append(dt)
    num_neighbors.append(dset[i].x_pos.shape[0])
    ship_types.append(group_of.get(dset.items[i][1], "unknown"))

In [ ]:
many_neighbors = [x[0] for x in filter(lambda x: x[1] > 10, neighbor_list)]
subset3 = Subset(dset,many_neighbors)
df_many = predict_dataframe(subset3, pl_module, model, scene, device,batch_size=512)


dt_list = []
num_neighbors = []
for i in many_neighbors:
        dt = _traj_datetime(dset, i)
        dt_list.append(dt)
        num_neighbors.append(dset[i].x_pos.shape[0])
df_many["time"] = dt_list
df_many["num_neighbors"] = num_neighbors
df_many["date"] = df_many["time"].dt.date
df_many["ship_type"] = ship_types


df_many = df_many[df_many["n"] == 30]



In [ ]:
df_many[]

In [ ]:
df_many[(df_many["num_neighbors"] > 20) & (df_many["date"] == pd.to_datetime("2023-05-18").date())][["num_neighbors","ship_type"]].value_counts().sort_index(ascending=False)

In [ ]:
arg_maxs = Counter(np.argmax(dset[i].is_ego) for i in range(len(dset)))




In [ ]:
df_many[["num_neighbors","date"]].value_counts().sort_index()#.plot(kind="bar", figsize=(12, 5), rot=45)

In [ ]:

df_chosen = df_many[df_many["num_neighbors"] > 24]

In [ ]:
df_chosen = df_chosen.drop_duplicates(subset="target_id", keep="first")


In [ ]:
fig, axes = plot_trajectory_grid(
    df_chosen,
    dset=dset,
    land=land,
    nrows=2, ncols=3,
    pad=300,
    suptitle="Example trajectories with many neighbors",
    hspace=-0.3,
    #figsize=(24, 10)
)
plt.show()
fig.savefig("figures/busy_trajectory_grid.pdf", bbox_inches="tight")


In [ ]:
df_many

In [ ]:
import glob

In [ ]:
sorted(glob.glob("lightning_logs/nereus_ablation/version_1[0-9]/best.ckpt"))
